# Data Enhancing – Thermodynamic Stability (Alexandria)

Raw database → one labeled row per unique composition → 142 composition descriptors (Magpie + stoichiometry + valence orbitals).

* **Source:** Alexandria PBE 3D (5 chunks, 500,000 entries)
* **Target:** is_stable (min E_above_hull <= 0.05 eV/atom)

In [1]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")
TASK = "stability"
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"

In [2]:
# Cell 1: Load the raw database
raw = pd.read_parquet(RAW / "alexandria/alexandria_pbe3d.parquet")
print(f"raw entries: {len(raw):,}")
raw.head()

raw entries: 500,000


,mat_id,formula,spg,nsites,e_above_hull,e_form,band_gap_ind,total_mag
0,agm005737469,Sr(Cd5Rh)2,69,13,0.048584,-0.146191,0.0,1.000000e-07
1,agm002510391,Li3CaPt,221,5,0.223922,-0.213921,0.0,-4.000000e-07
2,agm004513860,Nd2TiGa4Pt3,115,10,0.057553,-0.848516,0.0,1.400000e-06
3,agm003886200,SrMnBr2,12,4,0.938851,-0.654886,0.0,5.001196e+00
4,agm034649907,Na2Tb3SmSe6,12,12,0.006221,-1.731583,0.0,-3.700000e-06


In [3]:
# Cell 2: Simple stats
raw.describe(include='all').T.head(15)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
mat_id,500000,500000,agm005737469,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
formula,500000,436791,BC,72,NaN,NaN,NaN,NaN,NaN,NaN,NaN
spg,500000.0,NaN,NaN,NaN,112.048302,72.635988,1.0,44.0,123.0,166.0,230.0
nsites,500000.0,NaN,NaN,NaN,10.260142,9.841015,1.0,5.0,8.0,12.0,224.0
e_above_hull,500000.0,NaN,NaN,NaN,0.343198,0.505453,0.0,0.042934,0.147881,0.398464,5.497966
e_form,500000.0,NaN,NaN,NaN,-0.358949,0.958681,-5.733355,-0.704196,-0.198742,0.115413,5.438597
band_gap_ind,500000.0,NaN,NaN,NaN,0.184193,0.709594,0.0,0.0,0.0,0.0,8.9427
total_mag,500000.0,NaN,NaN,NaN,1.344243,4.333531,-110.89535,-0.0,0.000001,0.942558,144.36371


In [4]:
# Cell 3: One label per unique composition (reduced formula, polymorph rule)
import prepare_datasets as P
lab = P.labeled_stability()
print(f"unique compositions: {len(lab):,}")
print(lab["target"].value_counts().rename({0: "negative", 1: "positive"}))
lab.head()

unique compositions: 436,791
target
negative    308136
positive    128655
Name: count, dtype: int64


,formula,target,e_above_hull_min,n_polymorphs
0,Ac,1,0.008453,2
1,Ac(Ag2Au)2,0,0.524734,1
2,Ac(Ag2H3)3,0,0.141739,1
3,Ac(Ag3Cl4)2,0,0.122411,1
4,Ac(Ag3Hg)2,0,0.420779,1


In [5]:
# Cell 4: Composition featurisation (vectorised Magpie, identical to matminer)
from featurize import featurize_formulas, feature_names
examples = pd.Series(["LiFePO4", "Fe2O3", "GaAs", "NdFeB", "SrTiO3"])
X, ok = featurize_formulas(examples)
X.index = examples
print(len(feature_names()), "descriptors")
X[["MagpieData mean Electronegativity", "MagpieData range Electronegativity", "MagpieData mean CovalentRadius", "MagpieData mean NValence", "MagpieData mean GSmagmom", "0-norm", "frac d valence electrons"]]

142 descriptors


,MagpieData mean Electronegativity,MagpieData range Electronegativity,MagpieData mean CovalentRadius,MagpieData mean NValence,MagpieData mean GSmagmom,0-norm,frac d valence electrons
LiFePO4,2.680,2.46,90.142857,5.428571,0.301523,4.0,0.157895
Fe2O3,2.796,1.61,92.400000,6.800000,0.844265,2.0,0.352941
GaAs,1.995,0.37,120.500000,14.000000,0.000000,2.0,0.714286
NdFeB,1.670,0.90,139.000000,5.666667,0.703554,3.0,0.352941
SrTiO3,2.562,2.49,110.600000,4.800000,0.000005,3.0,0.083333


In [6]:
# Cell 5: Check against matminer's reference implementation
from matminer.featurizers.composition import ElementProperty, Stoichiometry, ValenceOrbital
from pymatgen.core import Composition
ep, st, vo = ElementProperty.from_preset("magpie"), Stoichiometry(), ValenceOrbital(props=["frac"])
ref = np.array([ep.featurize(Composition(f)) + st.featurize(Composition(f)) + vo.featurize(Composition(f)) for f in examples], float)
print("max |ours - matminer| =", np.nanmax(np.abs(X.values - ref)))

max |ours - matminer| = 2.2737367544323206e-13


In [7]:
# Cell 6: Featurise the whole labeled set and the GNoME unlabeled pool
import time
t = time.time()
feats, ok = featurize_formulas(lab['formula'])
print(f"featurised {ok.sum():,} compositions in {time.time()-t:.1f}s")
df = pd.read_parquet(PROC / "stability_features.parquet")
gn = pd.read_parquet(PROC / "unlabeled_gnome_features.parquet", columns=["formula"])
print(f"saved labeled table: {df.shape} | GNoME unlabeled pool: {len(gn):,}")

featurised 436,791 compositions in 32.9s


saved labeled table: (436791, 146) | GNoME unlabeled pool: 554,054
